# Do Java para o C

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Ler a entrada com scanf (cada variável com seu &) e imprimir com printf, usando largura e precisão como %7.2f.
- Escolher entre char, int, unsigned e double, e prever a divisão inteira, a aritmética com char e a volta dos unsigned.
- Escrever if, laços e switch do jeito do C, com break e com queda proposital (fall-through).

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. O formato de um programa em C
2. Do código-fonte ao programa rodando
3. Tipos: sem String, sem boolean
4. printf: imprimindo com formato
5. scanf e o &
6. Aritmética, e chars como números
7. if, laços e switch: quase como no Java
8. unsigned e a volta completa
9. Erros comuns

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## 1. O formato de um programa em C

Um programa em C não tem classe em volta: ele é um conjunto de **funções**, e começa no `main`. `#include <stdio.h>` traz as funções padrão de entrada e saída, como o `printf`. O `main` devolve um `int` para o sistema operacional: `0` quer dizer "tudo certo".

```c
#include <stdio.h>

int main(void) {
    printf("Olá!\n");
    return 0;
}
```

O `printf` não termina a linha sozinho: `\n` dentro do texto é a quebra de linha.

In [ ]:
#include <stdio.h>

int main(void) {
    printf("Olá do C!\n");
    printf("Mesma ");
    printf("linha\n");
    return 0;
}

🧪 **Experimente:** tire o `\n` do primeiro `printf` e rode. Nas próximas células, instruções soltas são rodadas dentro de um `main` para você, com os `#include` de costume.

## 2. Do código-fonte ao programa rodando

O Java compila para bytecode que a JVM roda. O C é compilado direto para **código de máquina** do seu processador: o compilador (`gcc` ou `clang`) transforma `hello.c` num programa que a CPU roda diretamente. O Sphinx compila cada célula assim, com os avisos ligados: um **aviso** (warning) é o compilador dizendo "isso compila, mas provavelmente é um bug". Trate avisos como erros.

#### 🎞️ Do código-fonte a um programa C rodando

![O hello.c é pré-processado, compilado e ligado pelo gcc em código de máquina, que a CPU roda direto, ao contrário do bytecode do Java na JVM](compile-steps.pt-br.svg)

In [ ]:
int apples = 3;
int pears = 4;
printf("%d frutas\n", apples + pears);

🧪 **Experimente:** apague um `;` e leia o erro do compilador. Depois acrescente uma linha `int unused = 1;` e leia o **aviso**: o programa ainda roda, mas o Sphinx o aponta.

## 3. Tipos: sem String, sem boolean

O C tem os tipos numéricos que você conhece (`int`, `long`, `double`, `char`), mas **não tem `String`** (texto é um array de `char`, numa unidade mais à frente) e, por padrão, **não tem `boolean`**: uma condição é um `int`, em que `0` é falso e qualquer outro valor é verdadeiro. `#include <stdbool.h>` acrescenta `bool`, `true` e `false` como nomes para isso.

O tamanho de cada tipo depende da máquina: o `sizeof` diz quantos bytes ele ocupa.

In [ ]:
printf("char: %zu byte\n", sizeof(char));
printf("int: %zu bytes\n", sizeof(int));
printf("long: %zu bytes\n", sizeof(long));
printf("double: %zu bytes\n", sizeof(double));

int adult = 20 >= 18;          // uma comparação dá 1 ou 0
printf("20 >= 18 dá %d, 5 > 9 dá %d\n", adult, 5 > 9);
if (42) {
    printf("qualquer valor diferente de zero conta como verdadeiro\n");
}

🧪 **Experimente:** mude `if (42)` para `if (0)` e para `if (-1)`. Quais imprimem?

## 4. printf: imprimindo com formato

O `printf` recebe um **texto de formato** com marcadores, e depois um valor por marcador, na ordem. O marcador precisa combinar com o tipo do valor:

| Marcador | Para | Exemplo |
|---|---|---|
| `%d` | `int` | `printf("%d", 42)` |
| `%ld` | `long` | `printf("%ld", 5000000000L)` |
| `%f`, `%.2f` | `double` (2 casas) | `printf("%.2f", 3.14159)` → `3.14` |
| `%c` | `char` | `printf("%c", 'A')` |
| `%s` | texto | `printf("%s", "Ada")` |
| `%zu` | um `sizeof` | `printf("%zu", sizeof(int))` |

Um número entre o `%` e a letra define a **largura**: `%5d` alinha à direita em 5 colunas, `%-5d` alinha à esquerda, `%7.2f` usa 7 colunas com 2 casas.

In [ ]:
int qty = 3;
double price = 4.5;
char size = 'M';
printf("%d camisetas, tamanho %c, a %.2f cada\n", qty, size, price);
printf("[%5d]\n[%-5d]\n", qty, qty);
printf("[%7.2f]\n", price * qty);
printf("%s tem %d letras\n", "Ada", 3);

🧪 **Experimente:** imprima `price` com `%d` e leia o aviso: o marcador não combina com o tipo, e a saída é lixo. Depois monte uma tabelinha com `%-10s %5d`.

## 5. scanf e o &

O `scanf` lê a entrada com os mesmos marcadores (mas `%lf` para um `double`). Ele precisa **escrever nas** suas variáveis, então recebe os **endereços** delas: `&qty` é "onde `qty` mora". Ele devolve quantos valores conseguiu ler, o que diz se a entrada era válida.

As células do notebook não leem o teclado, então abaixo o `sscanf` lê de um texto; ele funciona exatamente como o `scanf`.

In [ ]:
int qty;
double price;
int read = sscanf("3 2.50", "%d %lf", &qty, &price);   // como scanf("%d %lf", &qty, &price)
printf("leu %d valores: %d x %.2f = %.2f\n", read, qty, price, qty * price);

int n = 0;
int ok = sscanf("abc", "%d", &n);
printf("ler \"abc\" como número leu %d valores\n", ok);

🧪 **Experimente:** tire o `&` antes de `qty` e leia o aviso. Num programa de verdade esse `scanf` escreveria num lugar qualquer da memória. Volte com ele.

## 6. Aritmética, e chars como números

A aritmética funciona como no Java: `int / int` descarta as casas decimais (`7 / 2` é `3`), `%` é o resto, e `(double)` converte. Um `char` é um número pequeno, o código do caractere: `'A' + 2` é o código de `'C'`, e `c - '0'` transforma o caractere `'7'` no número 7. Imprima com `%c` para ver o caractere, ou `%d` para ver o código.

In [ ]:
printf("%d %d %.1f\n", 7 / 2, 7 % 2, (double) 7 / 2);
char c = 'A';
printf("%c tem código %d\n", c, c);
printf("%c\n", c + 2);
char digit = '7';
printf("como número: %d\n", digit - '0');

🧪 **Experimente:** imprima as letras de `'a'` a `'z'` com um laço `for (char ch = 'a'; ch <= 'z'; ch++)`. Quanto é `'a' - 'A'`?

### ✏️ Exercício: É maiúscula?

Devolva 1 quando o char for uma letra de A a Z, e 0 caso contrário. Por padrão, C não tem boolean: 1 é verdadeiro e 0 é falso. Um char é um número pequeno, então dá para compará-lo com `'A'` e `'Z'`.

In [ ]:
int is_upper(char c) {
    // TODO: compare c com a faixa
    return 0;
}

int main(void) {
    check_int("'A'", is_upper('A'), 1);
    check_int("'z'", is_upper('z'), 0);
    check_int("'5'", is_upper('5'), 0);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
int is_upper(char c) {
    return c >= 'A' && c <= 'Z';
}

int main(void) {
    check_int("'A'", is_upper('A'), 1);
    check_int("'z'", is_upper('z'), 0);
    check_int("'5'", is_upper('5'), 0);
    return 0;
}
```

</details>

## 7. if, laços e switch: quase como no Java

`if`/`else`, `for`, `while`, `do … while` e `switch` são iguaizinhos aos do Java. O C só tem o `switch` clássico, com `case …:` (sem as setas `->`), e como na forma clássica do Java, um `case` sem `break` **cai** no próximo. Às vezes isso é útil (vários casos dividindo o mesmo código), mas esquecer um `break` é um bug clássico.

In [ ]:
for (int qty = 1; qty <= 4; qty++) {
    printf("%d: ", qty);
    switch (qty) {
        case 1:
            printf("um só\n");
            break;
        case 2:
            printf("um par\n");
            break;
        default:
            printf("vários\n");
    }
}

🧪 **Experimente:** apague o primeiro `break` e rode: o que qty 1 imprime agora? O compilador pode avisar sobre a queda.

### ✏️ Exercício: Soma dos dígitos

Some os dígitos de um int não negativo: 1234 dá 10. O laço é igualzinho ao do Java.

In [ ]:
int digit_sum(int n) {
    // TODO: pegue n % 10 e depois divida por 10
    return 0;
}

int main(void) {
    check_int("1234", digit_sum(1234), 10);
    check_int("0", digit_sum(0), 0);
    check_int("909", digit_sum(909), 18);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
int digit_sum(int n) {
    int sum = 0;
    while (n > 0) {
        sum += n % 10;
        n /= 10;
    }
    return sum;
}

int main(void) {
    check_int("1234", digit_sum(1234), 10);
    check_int("0", digit_sum(0), 0);
    check_int("909", digit_sum(909), 18);
    return 0;
}
```

</details>

## 8. unsigned e a volta completa

Um tipo `unsigned` não guarda negativos, só de 0 até o máximo dele. Passar do máximo **dá a volta** para 0, como o hodômetro de um carro, e ficar abaixo de 0 volta para o máximo. Um `unsigned char` (1 byte) vai de 0 a 255. Já o estouro de um tipo com sinal é comportamento indefinido em C: evite.

In [ ]:
unsigned char counter = 254;
for (int i = 0; i < 4; i++) {
    printf("%u ", (unsigned) counter);
    counter++;
}
printf("\n");
unsigned int zero = 0;
printf("0 - 1 como unsigned: %u\n", zero - 1);

🧪 **Experimente:** comece o contador em 0 e conte **para baixo** 3 vezes com `counter--`. Para onde ele vai?

## 9. Erros comuns

- **Esquecer o `&` no `scanf`:** compila (talvez com aviso) e escreve onde não devia.
- **`=` no lugar de `==` numa condição:** em C **compila**, porque uma atribuição dá um número. `if (x = 5)` põe 5 em x e é sempre verdadeiro. Com os avisos ligados, o compilador aponta isso.
- **Um marcador que não combina com o tipo:** `%d` para um `double` imprime lixo.
- **Um `break` esquecido** num `switch`.
- **Divisão inteira:** `1 / 2` é `0`.

In [ ]:
int x = 3;
if (x == 5) {
    printf("cinco\n");
} else {
    printf("x vale %d\n", x);
}

🧪 **Experimente:** mude `x == 5` para `x = 5`, rode e leia o aviso: o que é impresso, e quanto vale x agora?

## Em resumo

O C não tem classes nem `String` ou `boolean` embutidos: um programa é feito de funções, começando em `int main(void)`. `#include <stdio.h>` traz o `scanf` e o `printf`. Uma string de formato diz o que ler ou imprimir: `%d` um `int`, `%f` um `double` (`%lf` no `scanf`), `%c` um `char`.

```c
#include <stdio.h>

int main(void) {
    int qty;
    double price;
    if (scanf("%d %lf", &qty, &price) != 2) {   // quantos valores ele leu
        printf("entrada inválida\n");
        return 1;
    }
    printf("%3d x %7.2f\n", qty, price);   // largura 3; largura 7 com 2 casas
    switch (qty) {
        case 1:  printf("single\n"); break;
        case 2:  printf("pair\n");   break;
        default: printf("many\n");
    }
    return 0;
}
```

Um `char` é um número pequeno (`'A' + 2` é `'C'`), `7 / 2` é `3`, e um valor `unsigned` dá a volta depois do seu máximo. Um `case` sem `break` cai no seguinte.

**Cuidado:** o `scanf` precisa do endereço de cada variável: `scanf("%d", &qty)`. Sem o `&` o programa compila, e depois escreve onde não devia.

## Leituras

- [Lecture 1: C](https://cs50.harvard.edu/x/2026/notes/1/) (CS50x (Harvard), lecture notes, 25 min): De "Hello, World" até "Types", "Format Codes" e "Loops and meow.c". O CS50 usa uma biblioteca própria: get_int e string vêm de cs50.h, que você não tem aqui. Leia a entrada com scanf, como no exemplo acima.
- [Input and output](https://www.freecodecamp.org/news/the-c-beginners-handbook/#heading-input-and-output) (freeCodeCamp, The C Beginner's Handbook, 6 min): Opcional: os especificadores de formato do printf e o scanf com &. "Unsigned integers" e "The problem with overflow", antes no mesmo livro, explicam a volta dos unsigned.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Linha do recibo** · Fácil
- **Códigos dos caracteres** · Fácil
- **Hodômetro** · Médio
- **Dia do ano** · Médio
- **Calculadora de linha de comando** · Difícil